# Stress test: complex branching pathways

This notebook builds a deliberately awkward DAG — fan-outs with a different
number of children at every node, a long spine, and multi-parent joins — then
checks the store's answers and timings hold up. Assertions throughout: if a
cell runs, the property it claims held.

In [1]:
import random
import shutil
import tempfile
import time
from pathlib import Path

from ancestree import LineageStore

workdir = Path(tempfile.mkdtemp(prefix="ancestree-stress-"))
store = LineageStore(workdir / "stress", gen_triggers=["spawn"])
rng = random.Random(42)

## Build the DAG

Breadth-first from a single root, five levels deep. The root always fans out;
every other node spawns between 0 and 4 children (seeded, so the run is
repeatable), and every level also gets one **join node with two parents** to
keep the graph honest about being a DAG rather than a tree.

In [2]:
started = time.perf_counter()
with store.create_node(step_type="spawn") as root:
    root.add_meta("level", 0)

frontier = [root]
created = 1
for level in range(1, 6):
    next_frontier = []
    for parent in frontier:
        fan_out = rng.randint(2, 4) if level == 1 else rng.randint(0, 4)
        for _ in range(fan_out):
            with store.create_node(step_type="spawn", parent=parent) as child:
                child.add_meta("level", level)
                (child / "payload.bin").write_bytes(rng.randbytes(2_000))
            next_frontier.append(child)
            created += 1
    if len(next_frontier) >= 2:  # a two-parent join at every level
        with store.create_node(step_type="join", parent=next_frontier[:2]) as join:
            join.add_meta("level", level)
        next_frontier.append(join)
        created += 1
    frontier = next_frontier

elapsed = time.perf_counter() - started
print(f"created {created} nodes in {elapsed:.2f}s ({elapsed / created * 1000:.1f} ms/node)")
assert store.stats()["nodes"] == created

created 189 nodes in 4.88s (25.8 ms/node)


## The shape of the thing

In [3]:
child_counts = {}
for node in store.find():
    count = len(store.children(node))
    child_counts[count] = child_counts.get(count, 0) + 1
print("children -> how many nodes have that many:")
for count in sorted(child_counts):
    print(f"  {count}: {'#' * child_counts[count]} ({child_counts[count]})")

joins = store.find(step_type="join")
assert all(len(j.parent_id) == 2 for j in joins)
print(f"{len(joins)} join nodes, all with exactly two parents")

by_generation = {}
for node in store.find():
    by_generation[node.generation] = by_generation.get(node.generation, 0) + 1
print("nodes per generation:", dict(sorted(by_generation.items())))

children -> how many nodes have that many:
  0: ################################################################################################################# (113)
  1: ################### (19)
  2: ################### (19)
  3: ################# (17)
  4: #################### (20)
  5: # (1)
5 join nodes, all with exactly two parents
nodes per generation: {0: 1, 1: 5, 2: 14, 3: 25, 4: 47, 5: 97}


## Deep queries under load

Lineage of the deepest node (which crosses the joins), filtered ancestry,
and a search across the whole store — with timings.

In [4]:
deepest = max(store.find(), key=lambda n: n.generation)
started = time.perf_counter()
chain = store.lineage(deepest)
lineage_ms = (time.perf_counter() - started) * 1000
print(f"deepest node {deepest.node_id}: lineage of {len(chain)} nodes in {lineage_ms:.1f} ms")
assert chain[-1] == deepest
positions = {n.node_id: i for i, n in enumerate(chain)}
assert all(
    positions[p] < positions[n.node_id]
    for n in chain for p in n.parent_id if p in positions
)
print("every node appears after all of its parents: OK")

started = time.perf_counter()
level_3 = store.find(level=3)
find_ms = (time.perf_counter() - started) * 1000
print(f"find(level=3) -> {len(level_3)} nodes in {find_ms:.1f} ms")

deep_joins = store.ancestors(deepest, step_type="join")
print(f"joins on the deepest path: {len(deep_joins)}")

deepest node 47b68a56: lineage of 6 nodes in 0.2 ms
every node appears after all of its parents: OK
find(level=3) -> 25 nodes in 0.6 ms
joins on the deepest path: 0


## Prune a whole branch, then reclaim the space

Dry run first (always), then for real. A join child with a surviving second
parent must NOT die with the pruned branch — that is the DAG-aware rule.

In [5]:
level_one = store.children(root)
assert level_one, "the root always fans out in this build"
victim = max(level_one, key=lambda n: len(store.prune(n)))
preview = store.prune(victim)  # dry run
print(f"pruning {victim.node_id} would remove {len(preview)} nodes")

before = store.stats()["nodes"]
started = time.perf_counter()
deleted = store.prune(victim, dry_run=False)
prune_ms = (time.perf_counter() - started) * 1000
print(f"deleted {len(deleted)} nodes in {prune_ms:.1f} ms")
assert store.stats()["nodes"] == before - len(deleted)

survivors = store.find(step_type="join")
assert all(store.get(j) is not None for j in survivors)
print(f"{len(survivors)} join nodes survived (their other parent lives on)")

reclaimed = store.compact()
print(f"compact() removed {reclaimed} orphaned chunks")
final = store.stats()
print("final stats:", {k: final[k] for k in ("nodes", "chunks", "dedup_ratio")})

store.close()
shutil.rmtree(workdir)
print("cleaned up")

pruning 1fe76314 would remove 71 nodes
deleted 71 nodes in 3.6 ms
5 join nodes survived (their other parent lives on)
compact() removed 71 orphaned chunks
final stats: {'nodes': 118, 'chunks': 112, 'dedup_ratio': 0.995}
cleaned up
